# Verification of the network code

Reproduces the checks of the section on implementation and verification.

> **LLM note:** the following cell only calls the modules named in it, whose own docstrings carry the `LLM-assisted` tags. The layout of this notebook and the notes above the cells were drafted with Claude Opus 5.5 (Claude Code, November 2026), code level 2, and are declared in the appendix of the report.

In [1]:
import os, tempfile
import numpy as np
from IPython.display import Image, display

os.environ["PREVIEW_DIR"] = tempfile.mkdtemp()   # the figure code also writes a PNG copy here, shown below
import figures
from plot_style import load_results, set_style

set_style()
RERUN = False    # True runs the experiment again; False draws the figure from the saved numbers in ../results/

def show(name):
    figures.FIGURES[name]()
    display(Image(os.path.join(os.environ["PREVIEW_DIR"], name + ".png"), width=850))


## Test suite

All tests: gradients against Autograd and finite differences, recovery of OLS, Ridge and logistic regression, numbers printed in the course material, and the polynomial results of the earlier report.

> **LLM note:** the following cell only calls the modules named in it, whose own docstrings carry the `LLM-assisted` tags. The layout of this notebook and the notes above the cells were drafted with Claude Opus 5.5 (Claude Code, November 2026), code level 2, and are declared in the appendix of the report.

In [2]:
import subprocess, sys
result = subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q', '-p', 'no:cacheprovider'], capture_output=True, text=True)
print(result.stdout[-400:])

_with_scikit_learn_on_digits
  C:\Users\rasul\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (60) reached and the optimization hasn't converged yet.
    warnings.warn(

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
142 passed, 4 warnings in 12.80s



## Sizes of the differences and cost of a gradient

`verification.py` writes `results/verification.json`. The cost of a gradient is measured in units of one forward pass.

> **LLM note:** the following cell only calls the modules named in it, whose own docstrings carry the `LLM-assisted` tags. The layout of this notebook and the notes above the cells were drafted with Claude Opus 5.5 (Claude Code, November 2026), code level 2, and are declared in the appendix of the report.

In [3]:
r = load_results('verification')
for name, row in r['gradients'].items():
    print(f"{name:30s} Autograd {row['autograd']:.1e}   finite differences {row['finite_differences']:.1e}")
print(r['closed_forms'])
for row in r['cost']:
    print(row['sizes'], row['n_parameters'], 'parameters: gradient =', round(row['ratio'], 2), 'forward passes')

mse_sigmoid                    Autograd 1.0e-16   finite differences 4.1e-10
mse_relu                       Autograd 2.9e-17   finite differences 2.0e-10
mse_leaky_relu                 Autograd 5.8e-17   finite differences 2.9e-10
cross_entropy_sigmoid          Autograd 1.0e-15   finite differences 4.6e-09
cross_entropy_relu             Autograd 1.2e-16   finite differences 3.3e-10
cross_entropy_leaky_relu       Autograd 1.5e-16   finite differences 4.3e-10
{'ols': 3.577696764890168e-15, 'ridge': 3.682625970262337e-15, 'logistic_probabilities': 7.781078192881097e-08}
[1, 50, 1] 151 parameters: gradient = 2.61 forward passes
[1, 100, 100, 1] 10401 parameters: gradient = 2.49 forward passes
[1, 100, 100, 100, 100, 1] 30601 parameters: gradient = 3.51 forward passes
[784, 128, 10] 101770 parameters: gradient = 3.04 forward passes
[784, 512, 10] 407050 parameters: gradient = 1.1 forward passes
[784, 512, 512, 10] 669706 parameters: gradient = 1.42 forward passes
